In [ ]:
# =========================================================
# CONFIGURAÇÃO LOCAL
# REMOVER ANTES DE SUBIR PARA O GITHUB
# =========================================================

import os

JAVA_HOME = (
    r"C:\Users\GCarapinadelima\Downloads"
    r"\microsoft-jdk-17.0.20.1-windows-x64"
    r"\jdk-17.0.20.1+1"
)

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = JAVA_HOME + r"\bin;" + os.environ["PATH"]


# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DOS GRÁFICOS
# ---------------------------------------------------------------------

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------
# CAMINHOS
# ---------------------------------------------------------------------

SCRIPT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]

OUTPUT_DIR = PROJECT_ROOT / "scripts" / "Analytics" / "outputs" / "gold_04"
GRAFICOS_DIR = PROJECT_ROOT / "scripts" / "Analytics" / "graficos" / "gold_04"

GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# FUNÇÕES AUXILIARES
# ---------------------------------------------------------------------

def carregar_csv(nome_arquivo):
    caminho = OUTPUT_DIR / nome_arquivo

    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    return pd.read_csv(caminho)


def formatar_tecnologia(valor):
    if pd.isna(valor):
        return ""

    valor = str(valor).strip()

    mapa = {
        "sql": "SQL",
        "python": "Python",
        "r": "R",
        "java": "Java",
        "javascript": "JavaScript",
        "scala": "Scala",
        "rust": "Rust",
        "julia": "Julia",
        "dax": "DAX",

        "c_c++_c#": "C / C++ / C#",
        "visual_basic_vba": "Visual Basic / VBA",

        "microsoft_powerbi": "Microsoft Power BI",
        "power_bi": "Power BI",
        "tableau": "Tableau",
        "looker": "Looker",
        "looker_studiogoogle_data_studio": "Looker Studio",
        "google_data_studio": "Looker Studio",
        "qlik_sense": "Qlik Sense",
        "qlik_view_qlik_sense": "Qlik",
        "metabase": "Metabase",
        "grafana": "Grafana",
        "amazon_quicksight": "Amazon QuickSight",

        "amazon_web_services_aws": "AWS",
        "aws": "AWS",
        "azure_microsoft": "Microsoft Azure",
        "microsoft_azure": "Microsoft Azure",
        "google_cloud_gcp": "Google Cloud",
        "gcp": "Google Cloud",
        "oracle_cloud": "Oracle Cloud",

        "cloud_propria": "Cloud própria",
        "servidores_on_premise_nao_utilizamos_cloud": "On-premise",

        "postgresql": "PostgreSQL",
        "sql_server": "SQL Server",
        "mysql": "MySQL",
        "mongodb": "MongoDB",
        "oracle": "Oracle",
        "snowflake": "Snowflake",
        "google_bigquery": "Google BigQuery",
        "amazon_redshift": "Amazon Redshift",
        "databricks": "Databricks",

        "scripts_python": "Scripts Python",
        "sql_&_stored_procedures": "SQL / Stored Procedures",
        "sql_stored_procedures": "SQL / Stored Procedures",
        "apache_airflow": "Apache Airflow",
        "airflow": "Apache Airflow",
        "aws_glue": "AWS Glue",
        "azure_data_factory": "Azure Data Factory",
        "google_cloud_dataflow": "Google Cloud Dataflow",
        "talend": "Talend",
        "pentaho": "Pentaho",
        "informatica": "Informatica"
    }

    valor_lower = valor.lower()

    if valor_lower in mapa:
        return mapa[valor_lower]

    return (
        valor
        .replace("_", " ")
        .strip()
        .title()
    )


def salvar_grafico(fig, nome):
    caminho = GRAFICOS_DIR / nome

    fig.savefig(
        caminho,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)


def ajustar_visual(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    ax.tick_params(
        axis="y",
        length=0
    )

    ax.grid(
        axis="x",
        alpha=0.15
    )

    ax.set_axisbelow(True)


def titulo_subtitulo(ax, titulo, subtitulo):
    ax.set_title(
        titulo,
        loc="left",
        fontsize=18,
        fontweight="bold",
        pad=28
    )

    ax.text(
        0,
        1.015,
        subtitulo,
        transform=ax.transAxes,
        fontsize=11,
        va="bottom"
    )


def grafico_barras(
    dados,
    titulo,
    subtitulo,
    arquivo,
    top_n=None
):
    dados = dados.copy()

    if top_n is not None:
        dados = (
            dados
            .sort_values("pct_adocao", ascending=False)
            .head(top_n)
        )

    dados["tecnologia"] = (
        dados["opcao"]
        .apply(formatar_tecnologia)
    )

    dados = (
        dados
        .sort_values("pct_adocao", ascending=True)
    )

    altura = max(
        6,
        len(dados) * 0.55
    )

    fig, ax = plt.subplots(
        figsize=(12, altura)
    )

    barras = ax.barh(
        dados["tecnologia"],
        dados["pct_adocao"]
    )

    maior_valor = dados["pct_adocao"].max()

    margem = max(
        maior_valor * 0.16,
        6
    )

    for barra, valor in zip(
        barras,
        dados["pct_adocao"]
    ):
        ax.text(
            valor + maior_valor * 0.015,
            barra.get_y() + barra.get_height() / 2,
            f"{valor:.1f}%",
            va="center",
            fontsize=10,
            fontweight="bold"
        )

    titulo_subtitulo(
        ax,
        titulo,
        subtitulo
    )

    ax.set_xlabel(
        "Adoção entre profissionais (%)",
        fontsize=10
    )

    ax.set_ylabel("")

    ax.set_xlim(
        0,
        maior_valor + margem
    )

    ajustar_visual(ax)

    plt.tight_layout()

    salvar_grafico(
        fig,
        arquivo
    )


def grafico_historico(
    dados,
    titulo,
    subtitulo,
    arquivo,
    top_n=5,
    edicao_referencia=None
):
    dados = dados.copy()

    dados["edicao"] = (
        dados["edicao"]
        .astype(str)
    )

    if edicao_referencia is None:
        edicao_referencia = dados["edicao"].max()

    referencia = (
        dados[
            dados["edicao"]
            == str(edicao_referencia)
        ]
        .sort_values("pct_adocao", ascending=False)
        .head(top_n)
    )

    tecnologias = referencia["opcao"].tolist()

    historico = (
        dados[
            dados["opcao"].isin(tecnologias)
        ]
        .copy()
    )

    edicoes = sorted(
        historico["edicao"]
        .unique()
        .tolist()
    )

    mapa_edicoes = {
        edicao: indice
        for indice, edicao in enumerate(edicoes)
    }

    fig, ax = plt.subplots(
        figsize=(12, 7)
    )

    for tecnologia in tecnologias:
        serie = (
            historico[
                historico["opcao"] == tecnologia
            ]
            .copy()
        )

        serie["ordem_edicao"] = (
            serie["edicao"]
            .map(mapa_edicoes)
        )

        serie = (
            serie
            .sort_values("ordem_edicao")
        )

        ax.plot(
            serie["ordem_edicao"],
            serie["pct_adocao"],
            marker="o",
            linewidth=2.2,
            markersize=7,
            label=formatar_tecnologia(tecnologia)
        )

        for _, linha in serie.iterrows():
            ax.annotate(
                f"{linha['pct_adocao']:.1f}%",
                (
                    linha["ordem_edicao"],
                    linha["pct_adocao"]
                ),
                textcoords="offset points",
                xytext=(0, 8),
                ha="center",
                fontsize=9
            )

    titulo_subtitulo(
        ax,
        titulo,
        subtitulo
    )

    ax.set_ylabel(
        "Adoção entre profissionais (%)",
        fontsize=10
    )

    ax.set_xlabel("")

    ax.set_xticks(
        range(len(edicoes))
    )

    ax.set_xticklabels(edicoes)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(
        axis="y",
        alpha=0.15
    )

    ax.legend(
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.01, 1)
    )

    plt.tight_layout()

    salvar_grafico(
        fig,
        arquivo
    )


def grafico_variacao(
    dados,
    titulo,
    subtitulo,
    arquivo
):
    dados = dados.copy()

    dados["tecnologia"] = (
        dados["opcao"]
        .apply(formatar_tecnologia)
    )

    dados["categoria_tecnologia"] = (
        dados["tecnologia"]
        + " | "
        + dados["categoria_nome"]
    )

    dados = (
        dados
        .sort_values("variacao_pp", ascending=True)
    )

    altura = max(
        7,
        len(dados) * 0.45
    )

    fig, ax = plt.subplots(
        figsize=(13, altura)
    )

    barras = ax.barh(
        dados["categoria_tecnologia"],
        dados["variacao_pp"]
    )

    limite = (
        dados["variacao_pp"]
        .abs()
        .max()
    )

    margem = max(
        limite * 0.18,
        2
    )

    for barra, valor in zip(
        barras,
        dados["variacao_pp"]
    ):
        if valor >= 0:
            x_texto = valor + limite * 0.02
            alinhamento = "left"

        else:
            x_texto = valor - limite * 0.02
            alinhamento = "right"

        ax.text(
            x_texto,
            barra.get_y() + barra.get_height() / 2,
            f"{valor:+.1f} p.p.",
            va="center",
            ha=alinhamento,
            fontsize=9,
            fontweight="bold"
        )

    ax.axvline(
        0,
        linewidth=1
    )

    titulo_subtitulo(
        ax,
        titulo,
        subtitulo
    )

    ax.set_xlabel(
        "Variação em pontos percentuais",
        fontsize=10
    )

    ax.set_ylabel("")

    ax.set_xlim(
        -limite - margem,
        limite + margem
    )

    ajustar_visual(ax)

    plt.tight_layout()

    salvar_grafico(
        fig,
        arquivo
    )


# ---------------------------------------------------------------------
# CARREGAMENTO DOS OUTPUTS
# ---------------------------------------------------------------------

linguagens_historico = carregar_csv("linguagens_programacao_historico.csv")
bi_historico = carregar_csv("ferramentas_bi_historico.csv")
cloud_historico = carregar_csv("cloud_historico.csv")
bancos_historico = carregar_csv("bancos_dados_historico.csv")
etl_engineer = carregar_csv("etl_data_engineer_historico.csv")
etl_analyst = carregar_csv("etl_data_analyst_historico.csv")
linguagem_preferida = carregar_csv("linguagem_preferida_2025_2026.csv")
maiores_crescimentos = carregar_csv("maiores_crescimentos_adocao.csv")
maiores_quedas = carregar_csv("maiores_quedas_adocao.csv")


# ---------------------------------------------------------------------
# LINGUAGENS DE PROGRAMAÇÃO - HISTÓRICO
# ---------------------------------------------------------------------

grafico_historico(
    dados=linguagens_historico,
    titulo=(
        "SQL e Python concentram a adoção entre "
        "as linguagens de programação"
    ),
    subtitulo=(
        "Evolução das cinco linguagens com maior adoção "
        "na última edição comparável"
    ),
    arquivo="01_linguagens_programacao_historico.png",
    top_n=5
)


# ---------------------------------------------------------------------
# LINGUAGEM PREFERIDA - 2025-2026
# ---------------------------------------------------------------------

grafico_barras(
    dados=linguagem_preferida,
    titulo=(
        "Python é a linguagem preferida pelos "
        "profissionais de dados"
    ),
    subtitulo=(
        "Percentual de respondentes por linguagem "
        "| edição 2025-2026"
    ),
    arquivo="02_linguagem_preferida_2025_2026.png"
)


# ---------------------------------------------------------------------
# FERRAMENTAS DE BI - HISTÓRICO
# ---------------------------------------------------------------------

grafico_historico(
    dados=bi_historico,
    titulo=(
        "Power BI mantém a liderança entre "
        "as ferramentas de Business Intelligence"
    ),
    subtitulo=(
        "Evolução das cinco ferramentas com maior adoção "
        "na edição mais recente"
    ),
    arquivo="03_ferramentas_bi_historico.png",
    top_n=5
)


# ---------------------------------------------------------------------
# CLOUD - HISTÓRICO
# ---------------------------------------------------------------------

grafico_historico(
    dados=cloud_historico,
    titulo=(
        "AWS lidera a adoção de Cloud "
        "entre os profissionais"
    ),
    subtitulo=(
        "Evolução das cinco opções com maior adoção "
        "na edição mais recente"
    ),
    arquivo="04_cloud_historico.png",
    top_n=5
)


# ---------------------------------------------------------------------
# BANCOS DE DADOS - HISTÓRICO
# ---------------------------------------------------------------------

grafico_historico(
    dados=bancos_historico,
    titulo=(
        "PostgreSQL e SQL Server estão entre "
        "os bancos de dados mais adotados"
    ),
    subtitulo=(
        "Evolução das cinco tecnologias com maior adoção "
        "na edição mais recente"
    ),
    arquivo="05_bancos_dados_historico.png",
    top_n=5
)


# ---------------------------------------------------------------------
# ETL - DATA ENGINEER
# ---------------------------------------------------------------------

grafico_historico(
    dados=etl_engineer,
    titulo=(
        "Scripts Python lideram as ferramentas de ETL "
        "entre Data Engineers"
    ),
    subtitulo=(
        "Evolução das cinco tecnologias com maior adoção "
        "na edição mais recente"
    ),
    arquivo="06_etl_data_engineer_historico.png",
    top_n=5
)


# ---------------------------------------------------------------------
# ETL - DATA ANALYST
# ---------------------------------------------------------------------

grafico_historico(
    dados=etl_analyst,
    titulo=(
        "Python e SQL concentram o uso de ETL "
        "entre Data Analysts"
    ),
    subtitulo=(
        "Evolução das cinco tecnologias com maior adoção "
        "na edição mais recente"
    ),
    arquivo="07_etl_data_analyst_historico.png",
    top_n=5
)


# ---------------------------------------------------------------------
# MAIORES CRESCIMENTOS DE ADOÇÃO
# ---------------------------------------------------------------------

grafico_variacao(
    dados=maiores_crescimentos,
    titulo="Tecnologias com maior crescimento de adoção",
    subtitulo=(
        "Variação entre a primeira e a última edição "
        "válida de cada tecnologia"
    ),
    arquivo="08_maiores_crescimentos_adocao.png"
)


# ---------------------------------------------------------------------
# MAIORES QUEDAS DE ADOÇÃO
# ---------------------------------------------------------------------

grafico_variacao(
    dados=maiores_quedas,
    titulo="Tecnologias com maior retração de adoção",
    subtitulo=(
        "Variação entre a primeira e a última edição "
        "válida de cada tecnologia"
    ),
    arquivo="09_maiores_quedas_adocao.png"
)